# Derivation of EWMA Variance Estimate's Recursive Formula
Define the EWMA variance estimate at time $t$ as an exponentially weighted average of past squared returns:
$$σ^2_t = (1 - λ)\sum_{i=1}^{\infty}λ^{i - 1}r^2_{t - i}\quad(1)$$

where $r_{t-i}$ is the log return $i$ periods before time $t$, $λ \in (0, 1)$ is the decay factor (RiskMetrics uses $λ = 0.94$ for daily data), and $(1 - λ)$ is a normalizing constant so the weights sum to 1.

Now consider the one-step back formula,
$$σ^2_{t - 1} = (1 - λ)\sum_{i=1}^{\infty}λ^{i - 1}r^2_{t - 1 - i}\quad(2)$$

In (1), we can isolate the first term in the sum:
$$σ^2_t = (1 - λ)r^2_{t - 1} + (1 - λ)\sum_{i=2}^{\infty}λ^{i - 1}r^2_{t - i}$$

And then reindex the sum:
$$σ^2_t = (1 - λ)r^2_{t - 1} + (1 - λ)\sum_{i=1}^{\infty}λ^{i}r^2_{t - 1 - i}$$

And then factor out a λ from the sum:
$$σ^2_t = (1 - λ)r^2_{t - 1} + λ(1 - λ)\sum_{i=1}^{\infty}λ^{i - 1}r^2_{t - 1 - i}$$

Notice the sum and the (1 - λ) factor are now just σ^2_{t - 1} from (2):
$$σ^2_t = λσ^2_{t - 1} + (1 - λ)r^2_{t - 1}\quad(3)$$

Which gives our desired final recursive formula (cross referenced with the fourth edition of "J.P. Morgan/Reuters, RiskMetrics — Technical Document, New York", p. 78-82, hosted on MSCI).

# EWMA's Relation To GARCH (Optional)
The EWMA model is a special, restricted case of the standard GARCH(1,1) model where the constant term is zero and the persistence weights sum to one, i.e. it is a simpler case.

The GARCH(1, 1) equation, without derivation, is as follows:
$$σ^2_t = \omega + {\alpha}r^2_{t - 1} + \betaσ^2_{t - 1}\quad(4)$$

In EWMA, $\omega = 0$ and $\alpha + \beta = 1$, unlike GARCH(1, 1) where it is required that $\omega > 0$ and $\alpha + \beta < 1$. In addition, GARCH(1, 1) allows our model to exhibit mean reversion, i.e. a pull of volatility towards a "long-run unconditional variance level", while EWMA simply assumes volatility follows a random walk. In terms of estimation, EWMA is much simpler (**and thus why it's the choice in the first major version of this project**) as it doesn't require complex parameter optimization because λ is chosen subjectively or preset by convention (e.g., 0.94 in RiskMetrics). On the other hand, GARCH(1, 1) requires statistical estimation (MLE) to fit all the parameters to the sample reference.

# Derivation of Parametric Equations for VaR and ES
## Preliminary Calculations and Assumptions
Let the terminal loss L at time horizon T be $L = P_0 - P_T$ where $P_T$ is
$$P_T = P_0e^{Δt[(μ - \frac{1}{2}σ^2) + σW]} = P_0e^R$$
i.e. prices are lognormal and returns R ~ $N(μ_R, σ_R^2)$ where $μ_R = Δt(μ - \frac{1}{2}σ^2)$ and $σ_R^2 = Δtσ^2$

For small returns, we can use a first-order Taylor approximation of the exponential function to approximate $P_T$ for small R, which is the case here given we are dealing with small daily log returns. Thus,
$$P_T \approx P_0(1 + R)$$

Now rewrite, $L = P_0 - P_T$:
$$L = P_0 - P_T \approx P_0 - P_0(1 + R) = -P_0R\quad(1)$$
## VaR
We begin by writing the probability that the loss exceeds the VaR (derived from the def. of VaR: $VaR_\alpha = inf\{x : \mathbb{P}(L \leq x) \geq \alpha\}$):
$$\mathbb{P}(L > VaR_\alpha) = 1 - \alpha$$

Substituting (1) into the above probability gives
$$\mathbb{P}(-P_0R > VaR_\alpha) = 1 - \alpha$$

Since $P_0$ is positive, dividing both sides of the inequality $−P_0R > VaR_α$ by $−P0$ reverses the inequality:
$$\mathbb{P}(R < -\frac{VaR_\alpha}{P_0}) = 1 - \alpha$$

Given that $R$ is normally distributed, we can standardize it using the z-score transformation:

$$Z = \frac{R - μ_R}{σ_R}$$

This allows the probability statement to be rewritten in terms of the standard normal distribution:
$$\mathbb{P}(Z < -\frac{\frac{VaR_\alpha}{P_0} - μ_R}{σ_R}) = 1 - \alpha$$

Let $z_{1−α}$ denote the z-score corresponding to the cumulative probability $1 − α$. Therefore, the equation becomes
